# Ders 05 - Ajanik RAG


## Kurulum

Bu not defteri, Microsoft Agent Framework kullanarak Agentic RAG (Retrieval-Augmented Generation) desenini göstermektedir.

**Önkoşullar:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — Azure AI Search hizmet uç noktanız
- `AZURE_SEARCH_API_KEY` — Azure AI Search API anahtarınız
- Ortam değişkenleri aracılığıyla yapılandırılmış Azure OpenAI dağıtımı
- Azure CLI kimlik doğrulaması yapılmış (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG Nedir?

Geleneksel RAG sabit bir işlem hattını takip eder: belgeleri getirir, sonra bir yanıt üretir. **Agentic RAG** ise ajana, bilgiyi **ne zaman** ve **nasıl** alacağı konusunda özerklik vererek bir adım ileri gider.

Agentic RAG ile ajan:
- Bir soruyu yanıtlamadan önce bilgiyi getirmenin gerekli olup olmadığına **karar verebilir**
- Hangi veri kaynağına veya araca sorgu yapılacağına **seçim yapabilir**
- Getirilen sonuçları **değerlendirir** ve ilk deneme yetersizse takip eden sorgular yapabilir
- Birden fazla bilgi getirme adımından gelen bilgileri tutarlı bir cevapta **birleştirebilir**

Bu, ajanın statik bir getir-sonra-üret işlem hattına kıyasla daha esnek ve doğru olmasını sağlar.


## Bir Arama Aracı Oluşturma

Agentic RAG'de, harici veri kaynakları, ajanın talep üzerine çağırabileceği **araçlar** olarak sarılır. Bu, ajanın sorgulamayı zorunlu bir adım yerine alabileceği diğer eylemlerden biri olarak işlemesine olanak tanır.

Aşağıda bir seyahat bilgi tabanı tanımlıyor ve ajanın destinasyon bilgilerini araştırmak için çağırabileceği bir araç olarak sunuyoruz.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG Ajanı Oluşturma

Şimdi, **cevap vermeden önce her zaman bilgi getirmesi** talimatı verilen bir ajan oluşturuyoruz. Ajan, yanıtlarını kendi eğitim verilerine güvenmek yerine bilgi tabanına dayandırmak için `search_travel_knowledge` aracını kullanır.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## İteratif Geri Getirme — Maker-Checker Deseni

Agentic RAG'in temel bir avantajı **iteratif geri getirme**dir. Ajan, ilk bulgularını doğrulamak, rafine etmek veya genişletmek için birden fazla arama turu gerçekleştirebilir — tıpkı bir "maker-checker" iş akışı gibi:

1. **Maker adımı**: Ajan başlangıç bilgilerini getirir ve bir yanıt taslağı hazırlar.
2. **Checker adımı**: Ajan detayları doğrulamak veya boşlukları doldurmak için ek geri getirmeler yapar.

Aşağıda, birden çok destinasyonun karşılaştırılmasını gerektiren bir soru sorulur ve ajan birkaç kez arama yapmaya yönlendirilir.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Özet

Bu derste, Microsoft Agent Framework kullanarak **Agentic RAG** sistemi nasıl kurulur öğrendiniz:

- **Agentic RAG**, ajanların bilgi alım zamanına otonom olarak karar vermesini sağlar, böylece bilgi alımı sabit değil, dinamik hale gelir.
- **Araçlar veri kaynakları olarak**: Dış bilgi tabanları (Azure AI Search gibi) ajanın çağırabileceği araçlar olarak sarılır.
- **Yinelemeli bilgi alımı**: Yapıcı-denetleyici deseni, ajanın nihai cevabı üretmeden önce birden fazla bilgi alım turu yapmasına — arama, doğrulama ve iyileştirme — olanak tanır.

Üretimde, büyük ölçekli seyahat belgesi alımlarını yönetmek için bellek içi `TRAVEL_KNOWLEDGE_BASE` yerine gerçek bir Azure AI Search dizini kullanırsınız.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, [Co-op Translator](https://github.com/Azure/co-op-translator) adlı yapay zeka çeviri hizmeti kullanılarak çevrilmiştir. Doğruluk için çaba gösterilse de, otomatik çevirilerde hatalar veya yanlışlıklar olabileceğini lütfen unutmayın. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucunda ortaya çıkabilecek yanlış anlamalar veya yorum farklılıklarından sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
